# Lab 09 — RBM and contrastive divergence

**Dataset:** [ylecun/mnist](https://huggingface.co/datasets/ylecun/mnist). Use binarized 0/1 images and a tiny RBM so energy, conditional sampling, and contrastive divergence remain inspectable.

**Workflow:** predict → implement → baseline → controlled Gibbs-step intervention → results table → conclusion.

## The math behind the experiment

An RBM assigns an energy to a visible vector $v$ and hidden vector $h$:

$$
E(v,h)=-a^Tv-b^Th-v^TWh.
$$

Lower energy means the configuration is more compatible with the learned model. Conditional probabilities are logistic functions, for example

$$
p(h_j=1\mid v)=\sigma\left(b_j+\sum_i v_iW_{ij}\right).
$$

Contrastive divergence approximates the difference between statistics under real data and statistics under a short Gibbs chain. That approximation is the reason the number of Gibbs steps becomes an important experimental variable.

In [ ]:
!pip -q install datasets torch pandas matplotlib
from datasets import load_dataset
import numpy as np,pandas as pd,torch
tr=load_dataset('ylecun/mnist',split='train[:1000]')
images=np.array([np.array(x) for x in tr['image']],dtype=np.float32)/255.
# Downsample each 28x28 image to 14x14; do not take the first 196 flattened pixels.
X=torch.tensor(images[:,::2,::2].reshape(len(images),-1),dtype=torch.float32)
X=(X>.5).float()
print(X.shape)

## 1. Prediction

Predict that learning will reduce reconstruction error relative to an untrained/random model. Predict that increasing Gibbs steps changes the sampling approximation and therefore the reconstruction statistics.

In [ ]:
class RBM:
    def __init__(self,v=196,h=32):
        self.W=torch.randn(v,h)*.01; self.a=torch.zeros(v); self.b=torch.zeros(h)
    def ph(self,v): return torch.sigmoid(v@self.W+self.b)
    def pv(self,h): return torch.sigmoid(h@self.W.T+self.a)
    def cd1(self,v):
        ph=self.ph(v); h=(ph>torch.rand_like(ph)).float(); pv=self.pv(h); vk=(pv>torch.rand_like(pv)).float(); phk=self.ph(vk)
        self.W += .05*(v.T@ph-vk.T@phk)/len(v); self.a += .05*(v-vk).mean(0); self.b += .05*(ph-phk).mean(0)
        return float(((v-vk)**2).mean())
torch.manual_seed(0)
rbm=RBM(); rows=[]
for epoch in range(15): rows.append([epoch+1,rbm.cd1(X)])
pd.DataFrame(rows,columns=['epoch','reconstruction_MSE'])

## 2. Controlled intervention: Gibbs transitions

Keep the trained RBM parameters and the initial data batch fixed. Change only the number of alternating hidden/visible Gibbs transitions. Each transition samples \(h\sim P(h\mid v)\), then \(v\sim P(v\mid h)\).

We compare the generated batch's hidden-visible association statistics with the data batch's statistics. This is more faithful to the contrastive-divergence idea than calling the distance from the starting image a "reconstruction error." The statistic gap is still only a diagnostic—not a likelihood estimate or proof that a longer chain is better.

In [ ]:
import time

with torch.no_grad():
    data_moment=(X.T @ rbm.ph(X))/len(X)

def run_steps(k,seed):
    # Reset the random stream so each chain length shares the same first transition.
    torch.manual_seed(seed)
    v=X.clone()
    start=time.perf_counter()
    with torch.no_grad():
        for _ in range(k):
            ph=rbm.ph(v)
            h=(ph>torch.rand_like(ph)).float()
            pv=rbm.pv(h)
            v=(pv>torch.rand_like(pv)).float()
        model_moment=(v.T @ rbm.ph(v))/len(v)
    elapsed=time.perf_counter()-start
    moment_gap=float(torch.linalg.norm(data_moment-model_moment).item())
    return {
        "seed":seed,
        "gibbs_transitions":k,
        "association_moment_gap":moment_gap,
        "visible_activation_mean":float(v.mean().item()),
        "sampling_seconds":elapsed
    }

raw_sampling=pd.DataFrame([run_steps(k,seed) for seed in range(5) for k in [1,2,5,10]])
results=raw_sampling.groupby("gibbs_transitions").agg(
    association_moment_gap_mean=("association_moment_gap","mean"),
    association_moment_gap_sd=("association_moment_gap","std"),
    visible_activation_mean=("visible_activation_mean","mean"),
    sampling_seconds_mean=("sampling_seconds","mean"),
    sampling_seconds_sd=("sampling_seconds","std")
).reset_index()
results

## Visualize the RBM — learning curve and sampling approximation

The first plot shows the reconstruction statistic recorded during contrastive-divergence updates. The second compares the gap between data and sampled hidden-visible association statistics as the chain length changes, and reports the sampling time. A longer chain costs more transitions but is not guaranteed to reduce this finite-sample gap monotonically.

In [ ]:
import matplotlib.pyplot as plt

training_curve = pd.DataFrame(rows, columns=["epoch", "reconstruction_MSE"])
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(training_curve["epoch"], training_curve["reconstruction_MSE"], marker="o")
axes[0].set_title("RBM reconstruction during training")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Reconstruction MSE")
axes[0].grid(alpha=.25)

axes[1].plot(results["gibbs_transitions"], results["association_moment_gap_mean"], yerr=results["association_moment_gap_sd"], marker="s",capsize=3,label="association-statistic gap (mean ± SD)")
axes[1].set_xlabel("Gibbs transitions")
axes[1].set_ylabel("Frobenius norm gap")
axes[1].set_title("Data vs sampled statistics")
axes[1].grid(alpha=.25)
axes[1].legend()
fig.tight_layout()
plt.show()

display(results)
plt.figure(figsize=(6,4))
plt.plot(results["gibbs_transitions"],results["sampling_seconds_mean"],yerr=results["sampling_seconds_sd"],marker="o",capsize=3)
plt.xlabel("Gibbs transitions")
plt.ylabel("Sampling time (seconds)")
plt.title("Sampling compute grows with chain length")
plt.grid(alpha=.25)
plt.tight_layout()
plt.show()

## Conclusion / answer key

The RBM should learn non-random reconstruction statistics, but reconstruction MSE is not a likelihood estimate. The Gibbs intervention tracks a data-vs-sampled association-statistic gap and the environment-specific time needed to generate samples. The gap need not decrease monotonically because sampling is stochastic and the model may not be well trained. Do not infer better likelihood from one statistic alone.

**Research extension:** compare CD-1 and CD-5 across at least three initialization seeds at a fixed wall-clock budget. Report variability, sample diversity, and a defensible distributional metric where feasible.